In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

MODEL = os.getenv("OLLAMA_MODEL")

from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.7
)

llm.invoke("Bonjour").content

"Bonjour ! Comment puis-je vous aider aujourd'hui ?"

In [4]:
memoire = {
    "aime": ["poulet", "pâtes"],
    "n_aime_pas": ["champignons"],
    "allergies": ["arachides"]
}

In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/recettes.txt")

documents = loader.load()

documents

C:\Users\Admin\AppData\Local\Temp\ipykernel_11176\3947998372.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'data/recettes.txt'}, page_content='Tajine de poulet aux olives\n\nIngrÃ©dients :\n- poulet\n- olives\n- oignon\n- citron confit\n\nPrÃ©paration :\nFaire revenir le poulet avec les oignons.\nAjouter les olives et le citron confit.\nLaisser cuire 45 minutes.\n\n====================================\n\nSpaghetti Bolognaise\n\nIngrÃ©dients :\n- pÃ¢tes\n- viande hachÃ©e\n- tomates\n- oignon\n\nPrÃ©paration :\nFaire cuire les pÃ¢tes.\nPrÃ©parer la sauce avec la viande et les tomates.\n\n====================================\n\nOmelette au fromage\n\nIngrÃ©dients :\n- Å“ufs\n- fromage\n- beurre\n\nPrÃ©paration :\nBattre les Å“ufs.\nAjouter le fromage.\nFaire cuire dans une poÃªle.\n\n\nEntrez les ingrÃ©dients disponibles :\npoulet, tomates, oignon, fromage')]

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

len(chunks)

2

In [7]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [8]:
import sys
print(sys.executable)

c:\Users\Admin\Desktop\Master_BDCC\IA_AGENTIQUE\.venv\Scripts\python.exe


In [9]:
from langchain_community.vectorstores import Chroma

In [10]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

len(chunks)

1

In [18]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

# embeddings locaux
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./db"
)

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Tu es un chef cuisinier personnel intelligent.

Utilise uniquement les recettes dans le contexte.

Contexte:
{context}

Question:
{question}

Réponds comme un chef clair et pratique :
- ingrédients
- étapes
- conseils
""")

In [21]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [22]:
response = rag_chain.invoke("Donne-moi une recette de couscous marocain")
print(response.content)

Bien sûr, je vais vous donner une recette de couscous marocain qui va vous faire déguster ce plat traditionnel marocain !

**Ingrédients :**

*   1 tasse d'eau
*   1 cuillère à café d'huile d'olive
*   2 gousses d'oignon finement hachées
*   1 poivron rouge finement haché
*   1 boîte de haricots verts égoutés et croustillants (environ 250 g)
*   1 cuillère à café de paprika
*   1 cuillère à café de curry en poudre
*   Sel et poivre, au goût
*   2 cuillères à soupe d'huile d'olive
*   2 cuillères à soupe de moutarde
*   1 cuillère à soupe de vinaigre de vin blanc
*   1/4 tasse de crème fraîche (facultatif)
*   Fines herbes fraîches (comme ciboulette, coriandre ou persil) pour la garniture

**Étapes :**

1.  Préparez l'eau et laissez-la chauffer à haute température.
2.  Dans une grande poêle, faites fondre l'huile d'olive à feu moyen. Ajoutez les gousses d'oignon finement hachées et le poivron rouge haché. Faites-les revenir pendant environ 5 minutes, jusqu'à ce qu'ils soient tendres et 

In [23]:
from langchain.tools import tool

@tool
def search_recipes(query: str):
    """Cherche des recettes dans la base vectorielle"""
    docs = vectorstore.as_retriever().invoke(query)
    return "\n\n".join([d.page_content for d in docs])

In [24]:
@tool
def shopping_list(recipe: str):
    """Génère une liste de courses simple à partir d'une recette"""
    return f"Liste de courses pour: {recipe}\n- ingrédients principaux à extraire de la recette"

In [25]:
@tool
def estimate_calories(dish: str):
    """Estime les calories d'un plat"""
    return f"{dish} contient environ 500-800 kcal (estimation simplifiée)"

In [30]:
plat = "lasagnes"

recette = search_recipes.invoke(plat)
courses = shopping_list.invoke(recette)
calories = estimate_calories.invoke(plat)

prompt = f"""
Pour le plat '{plat}' :

Recette :
{recette}

Liste de courses :
{courses}

Calories :
{calories}

Présente le résultat sous forme de fiche cuisine bien structurée.
"""

response = llm.invoke(prompt)

print(response.content)

**Fiche Cuisine : Lasagnes**

**Ingrédients Principaux**

* Poulet
* Tomates
* Oignon
* Fromage

**Recette : Spaghetti Bolognaise**

Ingrédients :

* Pâtes
* Viande hachée
* Tomates
* Oignon

Préparation :

1. Faire cuire les pâtes.
2. Préparer la sauce avec la viande et les tomates.

**Recette : Omelette au Fromage**

Ingrédients :

* Œufs
* Fromage
* Beurre

Préparation :

1. Battre les œufs.
2. Ajouter le fromage.
3. Faire cuire dans une poêle.

**Calories et Prévisions Nutriments**

Lasagnes contiennent environ 500-800 kcal (estimation simplifiée).

**Tajine de Poulet aux Olives**

Ingrédients :

* Poulet
* Olives
* Oignon
* Citron confit

Préparation :

1. Faire revenir le poulet avec les oignons.
2. Ajouter les olives et le citron confit.
3. Laisser cuire 45 minutes.

Note : Cette recette est une suggestion et peut varier en fonction des ingrédients et des préférences personnelles.
